# Semantic Model & Report Audit v2

Traces SQL → semantic model → visuals, computes reachability, emits `audit_report.xlsx`.

**Improvements over v1:**
- Fixed graph direction bug (tables now correctly reflect their columns' status)
- Resolves DAX table aliases (`d.foo` where `d` = `dim_customer`)
- Scans measure DAX for `Table[Column]` references (not just bare `[Column]`)
- Extracts filter/join column usage from downstream CTAS (plumbing detection)
- Filters out auto-generated `DateTableTemplate` / `LocalDateTable` noise
- Detects "broken visual references" (visual → measure that doesn't exist)
- Two-pass reachability: kills dead measures first, then re-audits columns

**Inputs required in the notebook folder:**
- `build.sql` — full CTAS pipeline
- `<model>.tmdl` — one per model (from Power BI Desktop → View → TMDL view)
- `<model>.pbix` — the report pbix (for thin reports, use the report pbix, not the model one)


## Cell 1 — Setup

In [ ]:
!pip install -q sqlglot networkx pandas openpyxl

In [ ]:
import re, json, zipfile
from pathlib import Path
from collections import defaultdict
import sqlglot
from sqlglot import exp
import networkx as nx
import pandas as pd

FOLDER = Path(".")
SQL_FILE = FOLDER / "build.sql"

MODELS = {
    "model_a": {
        "tmdl": FOLDER / "model_a.tmdl",
        "pbix": FOLDER / "model_a.pbix",
    },
    "model_b": {
        "tmdl": FOLDER / "model_b.tmdl",
        "pbix": FOLDER / "model_b_report.pbix",
    },
}

# Column-name patterns to hide from the dead list (auto-generated by Power BI)
IGNORE_PATTERNS = [
    r"DateTableTemplate_",
    r"LocalDateTable_",
]
_ignore_re = re.compile("|".join(IGNORE_PATTERNS))

G = nx.DiGraph()

def add(kind, name, **attrs):
    node = f"{kind}::{name}"
    if node not in G:
        G.add_node(node, kind=kind, name=name, **attrs)
    return node

def edge(src, dst, why):
    G.add_edge(src, dst, why=why)

for name, paths in MODELS.items():
    for k, p in paths.items():
        print(f"{'OK ' if p.exists() else 'MISS'}  {name}.{k}: {p}")
print(f"{'OK ' if SQL_FILE.exists() else 'MISS'}  sql: {SQL_FILE}")

## Cell 2 — Parse SQL, extract full lineage

**Key fix vs v1:** the "member of" edge now goes *from table to column* (table "owns"
its columns), so column reachability naturally propagates to table reachability.
Also captures every column referenced in WHERE / JOIN / GROUP BY / HAVING inside
every downstream CTAS — that's what proves a column is "plumbing" and not dead.

In [ ]:
sql_text = SQL_FILE.read_text(encoding="utf-8") if SQL_FILE.exists() else ""
sql_clean = re.sub(r"--[^\n]*", "", sql_text)
sql_clean = re.sub(r"/\*.*?\*/", "", sql_clean, flags=re.DOTALL)
batches = re.split(r"(?im)^\s*GO\s*$", sql_clean)

statements = []
parse_errors = []
for i, b in enumerate(batches):
    b = b.strip()
    if not b:
        continue
    try:
        for stmt in sqlglot.parse(b, read="tsql"):
            if stmt is not None:
                statements.append(stmt)
    except Exception as e:
        parse_errors.append((i, str(e)[:100]))

print(f"Batches: {len(batches)}, parsed statements: {len(statements)}, errors: {len(parse_errors)}")
for i, err in parse_errors[:3]:
    print(f"  batch {i}: {err}")

sql_tables = {}
for stmt in statements:
    if not isinstance(stmt, exp.Create):
        continue
    target = stmt.this
    target_name = target.this.name if isinstance(target, exp.Schema) else getattr(target, "name", None)
    if not target_name:
        continue
    tname = target_name.lower()
    select = stmt.expression
    if not isinstance(select, exp.Select):
        continue

    # Table node
    tbl_node = add("sql.tbl", tname)

    # Projected columns
    cols = []
    for proj in select.expressions:
        col_name = proj.alias_or_name.lower()
        if not col_name or col_name == "*":
            continue
        cols.append(col_name)
        col_node = add("sql.col", f"{tname}.{col_name}")
        # KEY FIX: table -> column (owns), not column -> table
        edge(tbl_node, col_node, "owns")

        # Projection edges: source columns feed this projected column
        for c in proj.find_all(exp.Column):
            src_tbl = (c.table or "").lower()
            src_col = c.name.lower()
            if src_tbl and src_tbl != tname:
                src_node = add("sql.col", f"{src_tbl}.{src_col}")
                edge(src_node, col_node, "projection")

    sql_tables[tname] = cols

    # Filter/join/group/having columns feed the target *table*
    # (this makes them "plumbing" — alive if the table is alive)
    def _add_ref_edges(clause, why):
        if clause is None: return
        for c in clause.find_all(exp.Column):
            src_tbl = (c.table or "").lower()
            src_col = c.name.lower()
            if src_tbl and src_tbl != tname:
                src_node = add("sql.col", f"{src_tbl}.{src_col}")
                edge(src_node, tbl_node, why)

    _add_ref_edges(select.args.get("where"), "filter")
    _add_ref_edges(select.args.get("group"), "group")
    _add_ref_edges(select.args.get("having"), "having")
    for j in select.find_all(exp.Join):
        _add_ref_edges(j, "join")

    # Also handle CTE definitions inside the SELECT
    with_clause = select.args.get("with")
    if with_clause:
        for cte in with_clause.expressions:
            cte_body = cte.this
            if isinstance(cte_body, exp.Select):
                _add_ref_edges(cte_body.args.get("where"), "cte_filter")
                _add_ref_edges(cte_body.args.get("group"), "cte_group")
                for j in cte_body.find_all(exp.Join):
                    _add_ref_edges(j, "cte_join")
                # CTE projected columns that reference source tables count too
                for proj in cte_body.expressions:
                    for c in proj.find_all(exp.Column):
                        src_tbl = (c.table or "").lower()
                        src_col = c.name.lower()
                        if src_tbl and src_tbl != tname:
                            src_node = add("sql.col", f"{src_tbl}.{src_col}")
                            edge(src_node, tbl_node, "cte_projection")

print(f"\nParsed {len(sql_tables)} SQL tables")
for t, cs in sorted(sql_tables.items()):
    print(f"  {t}: {len(cs)} cols")

## Cell 3 — Parse TMDL

**Key fix vs v1:** measure DAX now scanned for BOTH bare `[Column]` and qualified
`Table[Column]` references. Auto-generated date tables are filtered out.

In [ ]:
def parse_tmdl(path, model):
    if not path.exists():
        print(f"  {model}: TMDL missing, skipping")
        return {}, {}

    lines = path.read_text(encoding="utf-8").splitlines()
    columns = defaultdict(list)
    measures = {}
    current_table = None

    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        m = re.match(r"table\s+(['\"]?)(.+?)\1\s*$", stripped)
        if m:
            current_table = m.group(2)
            i += 1; continue

        m = re.match(r"column\s+(['\"]?)(.+?)\1\s*$", stripped)
        if m and current_table:
            columns[current_table].append(m.group(2))
            i += 1; continue

        m = re.match(r"measure\s+(['\"]?)(.+?)\1\s*=(.*)$", stripped)
        if m and current_table:
            mname = m.group(2)
            body = [m.group(3).strip()]
            measure_indent = len(line) - len(line.lstrip())
            i += 1
            while i < len(lines):
                nxt = lines[i]
                nxt_stripped = nxt.strip()
                nxt_indent = len(nxt) - len(nxt.lstrip())
                if not nxt_stripped:
                    i += 1; continue
                if nxt_indent > measure_indent:
                    if re.match(r"[a-zA-Z]+:\s", nxt_stripped) or nxt_stripped.startswith("annotation"):
                        i += 1; continue
                    body.append(nxt_stripped)
                    i += 1; continue
                break
            measures[mname] = " ".join(body).strip()
            continue

        i += 1

    # Filter out auto-generated tables
    columns = {t: cs for t, cs in columns.items() if not _ignore_re.search(t)}

    # Register nodes
    for tbl, cols in columns.items():
        tbl_node = add("model.tbl", f"{model}::{tbl}")
        for c in cols:
            col_node = add("model.col", f"{model}::{tbl}[{c}]")
            edge(tbl_node, col_node, "owns")
            sql_node = f"sql.col::{tbl.lower()}.{c.lower()}"
            if sql_node in G:
                edge(sql_node, col_node, "direct lake binding")

    # Register measures + scan DAX for column/measure refs
    for mname, dax in measures.items():
        mnode = add("model.measure", f"{model}::[{mname}]", dax=dax)

        # Qualified refs: Table[Column] or 'Table Name'[Column]
        for ref in re.finditer(r"(?:'([^']+)'|(\w+))\s*\[([^\]]+)\]", dax):
            tbl = ref.group(1) or ref.group(2)
            col = ref.group(3)
            # Skip functions like DATE[...] wait actually [Column] can only come from table
            if tbl in columns and col in columns[tbl]:
                edge(f"model.col::{model}::{tbl}[{col}]", mnode, "dax column ref")

        # Unqualified [Something] — either a measure or an ambiguous column
        for ref in re.finditer(r"(?<![\w'])\[([^\]]+)\]", dax):
            token = ref.group(1)
            if token in measures and token != mname:
                edge(f"model.measure::{model}::[{token}]", mnode, "dax measure ref")
                continue
            # try column in any table (Power BI resolves ambiguously in this case)
            for t, cs in columns.items():
                if token in cs:
                    edge(f"model.col::{model}::{t}[{token}]", mnode, "dax column ref (unqualified)")
                    break

    return columns, measures


model_columns = {}
model_measures = {}
for name, paths in MODELS.items():
    cols, meas = parse_tmdl(paths["tmdl"], name)
    model_columns[name] = cols
    model_measures[name] = meas
    total_cols = sum(len(v) for v in cols.values())
    print(f"{name}: {len(cols)} tables, {total_cols} columns, {len(meas)} measures")

## Cell 4 — sortByColumn edges

In [ ]:
def parse_sort_by(path, model, columns):
    if not path.exists(): return 0
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()
    current_table = None
    current_col = None
    added = 0
    for line in lines:
        s = line.strip()
        m = re.match(r"table\s+(['\"]?)(.+?)\1\s*$", s)
        if m: current_table = m.group(2); continue
        m = re.match(r"column\s+(['\"]?)(.+?)\1\s*$", s)
        if m: current_col = m.group(2); continue
        m = re.match(r"sortByColumn:\s*(['\"]?)(.+?)\1\s*$", s)
        if m and current_table and current_col:
            sort_col = m.group(2)
            disp_node = f"model.col::{model}::{current_table}[{current_col}]"
            sort_node = f"model.col::{model}::{current_table}[{sort_col}]"
            if disp_node in G and sort_node in G:
                edge(sort_node, disp_node, "sortByColumn")
                added += 1
    return added

for name, paths in MODELS.items():
    n = parse_sort_by(paths["tmdl"], name, model_columns.get(name, {}))
    print(f"{name}: {n} sortByColumn edges")

## Cell 5 — Extract PBIR visuals with alias resolution

**Key fix vs v1:** the walker now extracts the `From` clause of each visual's
prototypeQuery to build an alias table (`d` → `dim_customer`, `f` → `fact_sales`),
then resolves references through it. Also flags visual refs that don't match
any model column or measure — these are broken references worth surfacing.

In [ ]:
def walk_pbir_visuals(pbix_path, model):
    refs = []
    broken = []
    with zipfile.ZipFile(pbix_path) as z:
        names = z.namelist()

        # page id -> displayName
        page_names = {}
        for n in names:
            m = re.match(r"Report/definition/pages/([^/]+)/page\.json$", n)
            if m:
                try:
                    pj = json.loads(z.read(n).decode("utf-8"))
                    page_names[m.group(1)] = pj.get("displayName", m.group(1))
                except Exception:
                    page_names[m.group(1)] = m.group(1)

        # each visual
        for n in names:
            m = re.match(r"Report/definition/pages/([^/]+)/visuals/([^/]+)/visual\.json$", n)
            if not m: continue
            page = page_names.get(m.group(1), m.group(1))
            try:
                v = json.loads(z.read(n).decode("utf-8"))
            except Exception:
                continue

            vtype = (v.get("visual", {}) or {}).get("visualType", "?")
            query = (v.get("visual", {}) or {}).get("query", {}) or {}

            # Build alias table from prototypeQuery.From
            alias_to_table = {}
            for src in (query.get("queryDefinition", {}) or {}).get("From", []) or []:
                if "Name" in src and "Entity" in src:
                    alias_to_table[src["Name"]] = src["Entity"]

            def resolve(entity_or_alias):
                return alias_to_table.get(entity_or_alias, entity_or_alias)

            # queryState projections
            for role, block in (query.get("queryState", {}) or {}).items():
                for proj in block.get("projections", []) or []:
                    field = proj.get("field", {}) or {}
                    for entity, prop, kind in _extract_refs(field):
                        refs.append((page, vtype, resolve(entity), prop, kind))

            # Visual-level filters
            for fl in v.get("filterConfig", {}).get("filters", []) or []:
                for entity, prop, kind in _walk_generic(fl):
                    refs.append((page, f"{vtype}_filter", resolve(entity), prop, kind))

        # Page and report filters
        for n in names:
            m = re.match(r"Report/definition/pages/([^/]+)/page\.json$", n)
            if not m: continue
            try:
                pj = json.loads(z.read(n).decode("utf-8"))
            except Exception:
                continue
            page = page_names.get(m.group(1), m.group(1))
            for fl in pj.get("filterConfig", {}).get("filters", []) or []:
                for entity, prop, kind in _walk_generic(fl):
                    refs.append((page, "page_filter", entity, prop, kind))

        try:
            rj = json.loads(z.read("Report/definition/report.json").decode("utf-8"))
            for fl in rj.get("filterConfig", {}).get("filters", []) or []:
                for entity, prop, kind in _walk_generic(fl):
                    refs.append(("<report>", "report_filter", entity, prop, kind))
        except Exception:
            pass

    return refs


def _extract_refs(field):
    for kind in ("Column", "Measure", "HierarchyLevel", "Aggregation"):
        node = field.get(kind)
        if not node: continue
        # simple form
        expr = node.get("Expression", {}) or {}
        src = expr.get("SourceRef", {}) or {}
        entity = src.get("Entity") or src.get("Source")
        prop = node.get("Property")
        if entity and prop:
            yield entity, prop, kind
        # aggregation wraps a column
        if kind == "Aggregation":
            inner = node.get("Expression", {}).get("Column")
            if inner:
                src = (inner.get("Expression", {}) or {}).get("SourceRef", {}) or {}
                entity = src.get("Entity") or src.get("Source")
                prop = inner.get("Property")
                if entity and prop:
                    yield entity, prop, "Column"


def _walk_generic(obj):
    if isinstance(obj, dict):
        if obj.get("Property"):
            src = (obj.get("Expression", {}) or {}).get("SourceRef", {}) or {}
            entity = src.get("Entity") or src.get("Source")
            if entity:
                yield entity, obj["Property"], "unknown"
        for v in obj.values():
            yield from _walk_generic(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from _walk_generic(item)


usage_rows = []
broken_rows = []
for name, paths in MODELS.items():
    if not paths["pbix"].exists():
        print(f"{name}: pbix missing"); continue
    try:
        refs = walk_pbir_visuals(paths["pbix"], name)
    except Exception as e:
        print(f"{name}: walk failed ({e})"); continue

    for page, vtype, tbl, col, kind in refs:
        usage_rows.append((name, page, vtype, tbl, col, kind))
        vnode = add("visual", f"{name}/{page}/{vtype}/{tbl}.{col}")

        mnode = f"model.measure::{name}::[{col}]"
        cnode = f"model.col::{name}::{tbl}[{col}]"

        if mnode in G:
            edge(mnode, vnode, f"used in {vtype}")
        elif cnode in G:
            edge(cnode, vnode, f"used in {vtype}")
        else:
            # Try any table for the column (unqualified fallback)
            found = False
            for t, cs in model_columns.get(name, {}).items():
                if col in cs:
                    edge(f"model.col::{name}::{t}[{col}]", vnode, f"used in {vtype} (fuzzy)")
                    found = True
                    break
            if not found:
                broken_rows.append((name, page, vtype, tbl, col, kind))

    print(f"{name}: {len(refs)} refs, {sum(1 for r in broken_rows if r[0]==name)} broken")

usage_df = pd.DataFrame(usage_rows, columns=["model","page","visual","table","field","kind"])
broken_df = pd.DataFrame(broken_rows, columns=["model","page","visual","table","field","kind"])
print(f"\nTotal: {len(usage_df)} refs, {len(broken_df)} broken (visual points at nothing)")

## Cell 6 — Two-pass reachability

**Key fix vs v1:** first pass identifies dead measures. Then any column whose
only downstream is a dead measure is itself dead. This unmasks columns that
"look alive" but only feed dead code.

In [ ]:
def compute_reachability():
    visual_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "visual"]
    rev = G.reverse(copy=False)
    alive = set()
    for v in visual_nodes:
        alive.update(nx.descendants(rev, v))
        alive.add(v)
    return alive

# Pass 1: initial reachability
alive_pass1 = compute_reachability()

dead_measures = [n for n, d in G.nodes(data=True)
                 if d["kind"] == "model.measure" and n not in alive_pass1]

print(f"Pass 1: {len(dead_measures)} dead measures")

# Pass 2: prune edges FROM columns TO dead measures, recompute
edges_to_remove = []
for u, v, data in G.edges(data=True):
    if v in dead_measures and data.get("why", "").startswith("dax"):
        edges_to_remove.append((u, v))
for u, v in edges_to_remove:
    G.remove_edge(u, v)

alive_pass2 = compute_reachability()

# Build audit dataframe
rows = []
for n, d in G.nodes(data=True):
    if d["kind"] in ("visual", "visual.unresolved"):
        continue
    if _ignore_re.search(d["name"]):
        continue
    rows.append({
        "layer":     d["kind"],
        "object":    d["name"],
        "reachable": n in alive_pass2,
    })
audit = pd.DataFrame(rows).sort_values(["layer","reachable","object"])
print("\nReachability after 2 passes:")
print(audit.groupby(["layer","reachable"]).size().unstack(fill_value=0))

## Cell 7 — Export report

Adds two new sheets vs v1:
- `broken_refs`: visuals pointing at measures/columns that don't exist
- `dead_because_of_dead_measure`: columns kept alive only by dead measures

In [ ]:
dead = audit[~audit["reachable"]].copy()
alive_df = audit[audit["reachable"]].copy()

# Columns kept alive in pass 1 but killed by dead-measure pruning in pass 2
dead_by_cascade = [
    {"object": n, "layer": G.nodes[n]["kind"]}
    for n in alive_pass1 - alive_pass2
    if G.nodes[n]["kind"] in ("model.col", "sql.col")
]
cascade_df = pd.DataFrame(dead_by_cascade)

with pd.ExcelWriter("audit_report.xlsx") as w:
    dead.to_excel(w, sheet_name="dead_objects", index=False)
    alive_df.to_excel(w, sheet_name="alive_objects", index=False)
    usage_df.to_excel(w, sheet_name="visual_usage", index=False)
    if not broken_df.empty:
        broken_df.to_excel(w, sheet_name="broken_refs", index=False)
    if not cascade_df.empty:
        cascade_df.to_excel(w, sheet_name="dead_by_cascade", index=False)

print(f"Dead:  {len(dead)}")
print(f"Alive: {len(alive_df)}")
print(f"Broken visual refs: {len(broken_df)}")
print(f"Killed by dead-measure cascade: {len(cascade_df)}")
print("\nWritten to audit_report.xlsx")

## Cell 8 — Trace helper

Change `TARGET` to any node name to see its dependencies both ways.

In [ ]:
TARGET = "sql.col::dim_customer.region"

if TARGET in G:
    print(f"=== {TARGET} ===\n")
    print(f"Reachable from a visual? {TARGET in alive_pass2}")
    print("\nREADS FROM (upstream):")
    for pred in G.predecessors(TARGET):
        why = G.edges[pred, TARGET].get("why", "?")
        print(f"  <- {pred}  [{why}]")
    print("\nUSED BY (downstream):")
    for succ in G.successors(TARGET):
        why = G.edges[TARGET, succ].get("why", "?")
        print(f"  -> {succ}  [{why}]")
else:
    print(f"Node not in graph.")
    print("Available kinds:", sorted(set(d["kind"] for _, d in G.nodes(data=True))))

## Cell 9 — Cross-check the parser on itself

Sanity check: which measures does the graph believe reference each dropped column?
If nothing shows up here for a column but Finance actually uses it, the DAX parser
missed a reference — investigate before deleting.

In [ ]:
# For every dead sql.col, show what would be affected downstream
SAMPLE_LIMIT = 20

dead_sql_cols = [d["object"] for _, d in G.nodes(data=True)
                 if d["kind"] == "sql.col" and f"sql.col::{d['name']}" not in alive_pass2]

print(f"Sampling {min(SAMPLE_LIMIT, len(dead_sql_cols))} of {len(dead_sql_cols)} dead SQL columns:\n")
for name in dead_sql_cols[:SAMPLE_LIMIT]:
    node = f"sql.col::{name}"
    downs = list(G.successors(node))
    print(f"  {name:60s} -> {len(downs)} downstream nodes")
    for d in downs[:3]:
        print(f"     -> {d}")

---

## Reading the results

**dead_objects** — the hit list. Filter by `layer`:
- `sql.col` — drop from CTAS
- `model.col` — drop from TMDL (in Direct Lake models, also drop the SQL col)
- `model.measure` — delete the measure

**alive_objects** — sanity check that everything you *know* is used shows here.

**broken_refs** — **investigate first**. These are visuals pointing at fields
that don't exist in the model. Usually the fallout of prior cleanups where a
measure/column was deleted but the visual wasn't updated. Fix these before
trusting `dead_objects`, because a broken visual contributes nothing to
reachability.

**dead_by_cascade** — columns whose only surviving consumer is a dead measure.
Delete the measure and the column together.

**visual_usage** — every (page, visual, field) tuple. Useful for "where is X used?"

---

## Shipping deletions

1. Fix `broken_refs` first (rebind or delete the broken visual fields)
2. Re-run the notebook — the dead list should now be smaller and more trustworthy
3. Ship in order: **Report → Model → SQL**
4. After each layer, re-run the notebook. If nothing new turns up dead and
   nothing alive is now dead, the deletions were safe.

---

## Known remaining limitations

- **Bookmarks and drill-through**: not parsed. If a bookmark captures filter state
  on a field, that field will look dead. Rare but exists.
- **Field parameters / calculation groups**: partial support via unqualified
  reference matching, but complex cases can slip.
- **RLS DAX**: not parsed. Columns used only in role security will show as dead.
  Grep the TMDL for `role` blocks before deleting.
- **Runtime usage**: nothing checks what users actually query. Cross-reference
  Fabric workspace usage metrics if available.
